# Validate list_rides_across_shards
This notebook builds a tiny in-memory 3-shard setup, runs core.shard_queries.list_rides_across_shards, and compares it with a direct global baseline query to confirm correctness.

In [6]:
from __future__ import annotations

from contextlib import contextmanager
from datetime import datetime
from decimal import Decimal
from pathlib import Path
import os
import sys

from sqlalchemy import create_engine, select
from sqlalchemy.orm import sessionmaker
from sqlalchemy.pool import StaticPool

cwd = Path.cwd().resolve()
backend_dir = None
for candidate in (cwd, cwd.parent):
    if (candidate / "core").exists() and (candidate / "models").exists():
        backend_dir = candidate
        break

if backend_dir is None:
    raise RuntimeError("Run this notebook from backend/ or backend/notebooks/.")

if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

os.environ.setdefault("MYSQL_USER", "not_used")
os.environ.setdefault("MYSQL_PASSWORD", "not_used")
os.environ.setdefault("JWT_SECRET_KEY", "not_used")

import core.shard_queries as shard_queries
from models.member import Member
from models.ride import Ride

In [7]:
def make_in_memory_shards(shard_ids: tuple[int, ...] = (0, 1, 2)) -> dict[int, sessionmaker]:
    session_makers: dict[int, sessionmaker] = {}
    for shard_id in shard_ids:
        engine = create_engine(
            "sqlite+pysqlite://",
            connect_args={"check_same_thread": False},
            poolclass=StaticPool,
        )
        Member.__table__.create(bind=engine, checkfirst=True)
        Ride.__table__.create(bind=engine, checkfirst=True)
        session_makers[shard_id] = sessionmaker(bind=engine, autocommit=False, autoflush=False)
    return session_makers


def insert_member(db, member_id: int) -> None:
    db.add(
        Member(
            MemberID=member_id,
            OAUTH_TOKEN=f"oauth_{member_id}",
            Email=f"user_{member_id}@iitgn.ac.in",
            Full_Name=f"User {member_id}",
            Reputation_Score=Decimal("5.0"),
            Phone_Number=f"{member_id:010d}"[-10:],
            Gender="Other",
        )
    )


def seed_rides(session_makers: dict[int, sessionmaker]) -> None:
    with session_makers[0]() as db:
        insert_member(db, 1)
        db.add_all(
            [
                Ride(
                    RideID=1,
                    Host_MemberID=1,
                    Start_GeoHash="u09tun",
                    End_GeoHash="u09tvw",
                    Departure_Time=datetime(2026, 1, 1, 10, 0, 0),
                    Vehicle_Type="Sedan",
                    Max_Capacity=4,
                    Available_Seats=2,
                    Base_Fare_Per_KM=Decimal("10.00"),
                    Ride_Status="Open",
                ),
                Ride(
                    RideID=4,
                    Host_MemberID=1,
                    Start_GeoHash="u09tun",
                    End_GeoHash="u09tvw",
                    Departure_Time=datetime(2026, 1, 1, 9, 0, 0),
                    Vehicle_Type="Sedan",
                    Max_Capacity=4,
                    Available_Seats=0,
                    Base_Fare_Per_KM=Decimal("12.00"),
                    Ride_Status="Full",
                ),
            ]
        )
        db.commit()

    with session_makers[1]() as db:
        insert_member(db, 2)
        db.add_all(
            [
                Ride(
                    RideID=2,
                    Host_MemberID=2,
                    Start_GeoHash="u09tun",
                    End_GeoHash="u09tvw",
                    Departure_Time=datetime(2026, 1, 1, 8, 0, 0),
                    Vehicle_Type="Bike",
                    Max_Capacity=2,
                    Available_Seats=1,
                    Base_Fare_Per_KM=Decimal("8.00"),
                    Ride_Status="Open",
                ),
                Ride(
                    RideID=5,
                    Host_MemberID=2,
                    Start_GeoHash="u09tun",
                    End_GeoHash="u09tvw",
                    Departure_Time=datetime(2026, 1, 1, 11, 0, 0),
                    Vehicle_Type="SUV",
                    Max_Capacity=5,
                    Available_Seats=0,
                    Base_Fare_Per_KM=Decimal("15.00"),
                    Ride_Status="Full",
                ),
            ]
        )
        db.commit()

    with session_makers[2]() as db:
        insert_member(db, 3)
        db.add(
            Ride(
                RideID=6,
                Host_MemberID=3,
                Start_GeoHash="u09tun",
                End_GeoHash="u09tvw",
                Departure_Time=datetime(2026, 1, 1, 9, 30, 0),
                Vehicle_Type="Mini",
                Max_Capacity=3,
                Available_Seats=1,
                Base_Fare_Per_KM=Decimal("9.00"),
                Ride_Status="Open",
            )
        )
        db.commit()


@contextmanager
def patched_shards(session_makers: dict[int, sessionmaker]):
    original_makers = shard_queries.SHARD_SESSION_MAKERS
    original_ids = shard_queries.VALID_SHARD_IDS
    shard_queries.SHARD_SESSION_MAKERS = session_makers
    shard_queries.VALID_SHARD_IDS = tuple(sorted(session_makers.keys()))
    try:
        yield
    finally:
        shard_queries.SHARD_SESSION_MAKERS = original_makers
        shard_queries.VALID_SHARD_IDS = original_ids


def baseline_query(
    session_makers: dict[int, sessionmaker],
    statuses: tuple[str, ...],
    limit: int | None,
    order_desc: bool,
) -> list[Ride]:
    unique_statuses = tuple(dict.fromkeys(statuses))
    rides: list[Ride] = []

    for shard_id in sorted(session_makers.keys()):
        with session_makers[shard_id]() as db:
            stmt = select(Ride).where(Ride.Ride_Status.in_(unique_statuses))
            rides.extend(list(db.scalars(stmt)))

    rides.sort(key=lambda ride: (ride.Departure_Time, ride.RideID), reverse=order_desc)
    if limit is not None:
        return rides[:limit]
    return rides

In [8]:
session_makers = make_in_memory_shards()
seed_rides(session_makers)

with patched_shards(session_makers):
    actual_asc = shard_queries.list_rides_across_shards(
        statuses=("Open", "Full", "Open"),
        limit=3,
        order_desc=False,
    )

expected_asc = baseline_query(
    session_makers,
    statuses=("Open", "Full"),
    limit=3,
    order_desc=False,
)

assert [ride.RideID for ride in actual_asc] == [ride.RideID for ride in expected_asc]
assert [ride.RideID for ride in actual_asc] == [2, 4, 6]
print("Check 1 passed: merged and globally sorted with limit=3 -> [2, 4, 6]")

with patched_shards(session_makers):
    actual_desc = shard_queries.list_rides_across_shards(
        statuses=("Open", "Full"),
        limit=None,
        order_desc=True,
    )

expected_desc = baseline_query(
    session_makers,
    statuses=("Open", "Full"),
    limit=None,
    order_desc=True,
)

assert [ride.RideID for ride in actual_desc] == [ride.RideID for ride in expected_desc]
assert [ride.RideID for ride in actual_desc] == [5, 1, 6, 4, 2]
print("Check 2 passed: descending order works globally across shards.")

with patched_shards(session_makers):
    open_only = shard_queries.list_rides_across_shards(
        statuses=("Open",),
        limit=None,
        order_desc=False,
    )
    empty = shard_queries.list_rides_across_shards(
        statuses=(),
        limit=10,
        order_desc=False,
    )

assert [ride.RideID for ride in open_only] == [2, 6, 1]
assert empty == []
print("Check 3 passed: status filtering and empty-status fast return are correct.")

Check 1 passed: merged and globally sorted with limit=3 -> [2, 4, 6]
Check 2 passed: descending order works globally across shards.
Check 3 passed: status filtering and empty-status fast return are correct.


In [9]:
with patched_shards(session_makers):
    merged_preview = shard_queries.list_rides_across_shards(
        statuses=("Open", "Full"),
        limit=None,
        order_desc=False,
    )

[
    {
        "ride_id": ride.RideID,
        "status": ride.Ride_Status,
        "departure_time": str(ride.Departure_Time),
    }
    for ride in merged_preview
]

[{'ride_id': 2, 'status': 'Open', 'departure_time': '2026-01-01 08:00:00'},
 {'ride_id': 4, 'status': 'Full', 'departure_time': '2026-01-01 09:00:00'},
 {'ride_id': 6, 'status': 'Open', 'departure_time': '2026-01-01 09:30:00'},
 {'ride_id': 1, 'status': 'Open', 'departure_time': '2026-01-01 10:00:00'},
 {'ride_id': 5, 'status': 'Full', 'departure_time': '2026-01-01 11:00:00'}]

## Routes Using Cross-Shard Query Helpers

The following API routes use helpers from core/shard_queries.py for querying across multiple shards:

- GET /api/v1/rides (api/routes/rides.py: uses list_rides_across_shards)
- GET /api/v1/admin/rides/active (api/routes/admin.py: uses list_rides_across_shards)
- GET /api/v1/admin/rides/open (api/routes/admin.py: uses list_rides_across_shards)
- GET /api/v1/admin/rides/completed (api/routes/admin.py: uses list_rides_across_shards)
- GET /api/v1/admin/rides/stats (api/routes/admin.py: uses aggregate_ride_booking_stats_across_shards)